In [1]:
# import essential
import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))



In [8]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np

from aind_dynamic_foraging_models.generative_model import ForagerCollection


def fit_ctt_from_nwb(
    nwb: Any,
    *,
    number_of_learning_rate: int = 1,
    reset_to_threshold: bool = False,
    include_stay_bias: bool = True,
    fix_threshold: bool = True,
    threshold_fixed: float = 0.0,
    clamp_params: Optional[Dict[str, Any]] = None,
    workers: int = 4,
    seed: int = 42,
) -> Any:
    """
    Fit ForagerCompareThreshold from an in-memory NWB object and return the fitting_result.

    Inputs
    ------
    nwb : in-memory NWB object (already loaded)
    Parameters : model hyperparameters + fitting settings

    Returns
    -------
    fitting_result : object returned by forager.fit(...) (contains LPT/AIC/BIC/x/etc.)
    """
    df = nwb.trials.to_dataframe()

    choice = df.animal_response.map({0: 0, 1: 1, 2: np.nan}).to_numpy(dtype=float)
    reward = (df.rewarded_historyL | df.rewarded_historyR).to_numpy(dtype=bool)

    keep = ~np.isnan(choice)
    choice_valid = choice[keep].astype(int)
    reward_valid = reward[keep].astype(int)

    fc = ForagerCollection()
    forager = fc.get_forager(
        agent_class_name="ForagerCompareThreshold",
        agent_kwargs={
            "number_of_learning_rate": int(number_of_learning_rate),
            "reset_to_threshold": bool(reset_to_threshold),
            "include_stay_bias": bool(include_stay_bias),
            "fix_threshold": bool(fix_threshold),
            "threshold_fixed": float(threshold_fixed),
        },
    )

    DE_kwargs = dict(
        workers=int(workers),
        disp=False,
        seed=np.random.default_rng(seed),
    )

    forager.fit(
        choice_valid,
        reward_valid,
        clamp_params=clamp_params or {},
        DE_kwargs=DE_kwargs,
    )
    return forager

In [20]:
fitting_result.get_fitting_result_dict()['fitted_latent_variables']['value']

[0.0,
 0.27843589610274017,
 0.4793452439669444,
 0.6243142175231617,
 0.7289186250201594,
 0.8043974105794343,
 0.8588601928447659,
 0.6197226854230613,
 0.4471696441720947,
 0.6010974596898342,
 0.7121662459587547,
 0.7923094951938445,
 0.8501379870115768,
 0.8918649508897689,
 0.9219736301888899,
 0.6652630762841499,
 0.7584658515976472,
 0.5472817325447283,
 0.3948988529229767,
 0.5633807330421636,
 0.6849512098932903,
 0.7726721020827361,
 0.8359683490484817,
 0.8816406487703796,
 0.6361602446893978,
 0.7374662929971071,
 0.5321292048608917,
 0.3839653328630107,
 0.2770556013349114,
 0.478349272809682,
 0.34515966438482404,
 0.5274907200360547,
 0.6590542648196763,
 0.7539857961170173,
 0.8224849814291778,
 0.5934756385938991,
 0.706666613449601,
 0.7883411577906129,
 0.847274577189252,
 0.8897988171472325,
 0.6420468861436845,
 0.7417138821630339,
 0.8136300088338675,
 0.5870862082281292,
 0.7020562298533103,
 0.5065785743795925,
 0.3655289150757621,
 0.54218844015792,
 0.6696596

In [16]:
fitting_result.get_fitting_result_dict()['fitted_latent_variables'].keys()

dict_keys(['value', 'threshold', 'exploiting', 'choice_kernel', 'choice_prob', 'p_exploit', 'stay_bias'])

In [10]:
dir(fitting_result)

['ParamFitBoundModel',
 'ParamModel',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_cost_func_for_DE',
 '_fitting_result_to_dict',
 '_get_alpha_for_trial',
 '_get_params_list',
 '_get_params_model',
 '_get_stay_bias_value',
 '_get_threshold_value',
 '_optimize_DE',
 '_reset',
 'act',
 'agent_kwargs',
 'choice_history',
 'choice_kernel',
 'choice_prob',
 'current_option',
 'exploiting',
 'fit',
 'fitting_result',
 'fitting_result_cross_validation',
 'get_agent_alias',
 'get_choice_history',
 'get_fitting_result_dict',
 'get_latent_variables',
 'get_p_reward',
 'get_params',
 'get_params_str',
 'get_reward_history',
 'learn',
 'load',
 'n_actions',
 'n_trials',
 'par

In [9]:
fitting_result=fit_ctt_from_nwb(nwb=nwb_data)

2026-02-25 05:28:39,507 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...


In [4]:
from nwb_utils import NWBUtils

nwb_full_path='/root/capsule/data/behavior_nwb/753125_2024-10-14_15-37-15.nwb'

nwb_data=NWBUtils.read_behavior_nwb(nwb_full_path=nwb_full_path)

Found behavior NWB: /root/capsule/data/behavior_nwb/753125_2024-10-14_15-37-15.nwb


/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:620: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.6.0-alpha, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


Successfully read behavior NWB from: /root/capsule/data/behavior_nwb/753125_2024-10-14_15-37-15.nwb


In [ ]:
from behavior_utils import generate_behavior_summary_combined
from general_utils import find_ephys_sessions
from general_utils import smart_read_csv
import pandas as pd
from ephys_behavior import (
    get_the_mean_firing_rate_combined,   # ← updated name here
    correlate_firing_latent_multiple_variable
)
import os

# 1) find your sorted sessions
_, _, spike_sorted_sessions = find_ephys_sessions()
print("Sessions to process:", spike_sorted_sessions)


# 3) ensure results folder
results_folder = '/root/capsule/scratch/behavior_summary'
os.makedirs(results_folder, exist_ok=True)

# 4) iterate sessions
for sess in spike_sorted_sessions:
    try:
        print(f"\n=== Processing session {sess} ===")

        # a) behavior summary
        generate_behavior_summary_combined(
            session_names=[sess],
            save_result=True,
            save_folder=results_folder,
            save_name=f"behavior_summary-{sess}.csv"
        )

    except Exception as e:
        print(f"!! Error in session {sess}: {e}")
        continue


In [ ]:
from general_visualization import plot_behavior_session
from nwb_utils import NWBUtils
from behavior_utils import get_fitted_model_names
from general_utils import find_behavior_sessions

import os

folder_path = '/root/capsule/data/behavior_nwb'  # Replace with your folder path

# List only files (exclude subdirectories)
behavior_sessions = [f for f in os.listdir(folder_path)
         if os.path.isfile(os.path.join(folder_path, f))]


behavior_sessions=['ecephys_795396_2025-09-20_13-11-19_sorted_2025-10-29_20-43-11']
for behavior_session in behavior_sessions:
        nwb_data=NWBUtils.read_behavior_nwb(session_name=behavior_session)
        plot_behavior_session(nwb_data=nwb_data,model_alias='ForagingCompareThreshold',latent_name='right_choice_probability')

In [ ]:
nwb_data=NWBUtils.read_behavior_nwb(session_name=behavior_session)